# Small-grid macOS comparison

Fresh Apple Silicon results for the small inverter grid. The notebook compares DPsim native, FMI ME, and FMI CS against the native DPsim baseline, and also compares the Simulink native and imported-FMU CS replay at the same timesteps. The saved traces are under `build-fmi/small-inverter-grid/results/`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "dpsim-fmi").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from a DPsim checkout")
BUILD = ROOT / "build-fmi"
if not BUILD.exists():
    BUILD = ROOT / "build-macos-fmi-latest"
DATA = BUILD / "small-inverter-grid" / "results"
sweep = json.loads((DATA / "metrics_sweep.json").read_text())
steps = [50, 100, 500, 1000]
records = {}
for us in steps:
    key = f"dt_{us}us"
    traces = {
        "DPsim native": pd.read_csv(
            DATA / "dpsim" / "native" / key / "dpsim_native.csv"
        ),
        "DPsim ME": pd.read_csv(DATA / "dpsim" / "me" / key / "dpsim_me.csv"),
        "DPsim CS": pd.read_csv(DATA / "dpsim" / "cs" / key / "dpsim_cs.csv"),
        "Simulink native": pd.read_csv(
            DATA / "simulink" / "native" / key / "simulink_native.csv"
        ),
        "Simulink CS": pd.read_csv(DATA / "simulink" / "cs" / key / "simulink_cs.csv"),
    }
    for name, frame in traces.items():
        if not np.isfinite(frame.select_dtypes(include=[np.number]).to_numpy()).all():
            raise ValueError(f"non-finite values in {name} at {us} µs")
    records[us] = traces
print("macOS results:", DATA)

## Error and timing summary

In [ ]:
rows = []
for us in steps:
    tr = records[us]
    for method, reference, label in (
        ("DPsim ME", "DPsim native", "DPsim ME vs native"),
        ("DPsim CS", "DPsim native", "DPsim CS vs native"),
        ("Simulink CS", "Simulink native", "Simulink CS vs native"),
    ):
        for prefix, unit, quantity in (
            ("v", "V", "PCC voltage"),
            ("i", "A", "Inverter current"),
        ):
            cols = [f"{prefix}_{p}_{unit}" for p in "abc"]
            a = tr[method][cols].to_numpy()
            b = tr[reference][cols].to_numpy()
            rms = float(np.sqrt(np.mean((a - b) ** 2)))
            denom = float(np.sqrt(np.mean(b**2)))
            rows.append(
                {
                    "timestep [µs]": us,
                    "comparison": label,
                    "quantity": quantity,
                    "RMS error [unit]": rms,
                    "RMS error [% reference RMS]": 100 * rms / denom,
                }
            )
errors = pd.DataFrame(rows)
display(
    errors.pivot_table(
        index=["timestep [µs]", "comparison"],
        columns="quantity",
        values="RMS error [% reference RMS]",
    ).style.format("{:.6g}")
)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for comparison, group in errors.groupby("comparison"):
    for ax, quantity in zip(axes, ("PCC voltage", "Inverter current")):
        part = group[group.quantity == quantity]
        ax.plot(
            part["timestep [µs]"],
            part["RMS error [% reference RMS]"],
            "o-",
            label=comparison,
        )
for ax, title in zip(axes, ("PCC voltage error", "Inverter current error")):
    ax.set_xscale("log")
    ax.set_xlabel("Fixed timestep [µs]")
    ax.set_ylabel("RMS error [%]")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.show()

us = 100
tr = records[us]
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True, constrained_layout=True)
for name, style in (
    ("DPsim native", "k--"),
    ("DPsim ME", "b-"),
    ("DPsim CS", "orange"),
):
    axes[0].plot(tr[name].time_s, tr[name].v_a_V / 1000, style, label=name)
    axes[1].plot(tr[name].time_s, tr[name].i_a_A, style, label=name)
axes[0].set_ylabel("PCC phase-a voltage [kV]")
axes[1].set_ylabel("Inverter phase-a current [A]")
axes[1].set_xlabel("Time [s]")
for ax in axes:
    ax.axvline(2.5, color="0.3", ls=":")
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.suptitle("Small grid macOS response at 100 µs")
plt.show()